In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# =============================================================================
# INPUT FILES
# =============================================================================

HAZARD_CSV = Path("data") / "hazard.csv"
EXPOSURE_CSV = Path("data") / "exposure_district.csv"
VULNERABILITY_CSV = Path("data") / "vulnerability.csv"
GOVERNMENT_RESPONSE_CSV = Path("data") / "government_response_district.csv"
MASTER_CSV = Path("data") / "MASTER_VARIABLES.csv"

hazard = pd.read_csv(HAZARD_CSV)
exposure = pd.read_csv(EXPOSURE_CSV)
vulnerability = pd.read_csv(VULNERABILITY_CSV)
government_response = pd.read_csv(GOVERNMENT_RESPONSE_CSV)

# =============================================================================
# STANDARDIZE COLUMN NAMES
# =============================================================================

hazard.columns = hazard.columns.str.replace("_", "-", regex=False)
exposure.columns = exposure.columns.str.replace("_", "-", regex=False)
vulnerability.columns = vulnerability.columns.str.replace("_", "-", regex=False)
government_response.columns = government_response.columns.str.replace("_", "-", regex=False)

government_response = government_response.rename(
    columns={"district": "dtname"}
)

# =============================================================================
# KEEP REQUIRED COLUMNS
# =============================================================================

hazard = hazard[
    ["dtname", "timeperiod", "heat-hazard", "heat-days-score"]
]

exposure = exposure[
    ["dtname", "timeperiod", "exposure"]
]

vulnerability = vulnerability[
    ["dtname", "timeperiod", "vulnerability"]
]

government_response = government_response[
    [
        "dtname",
        "timeperiod",
        "government-response",
        "total-tender-awarded-value-fy-cumsum",
    ]
]

# =============================================================================
# MERGE COMPONENTS
# =============================================================================

df = hazard.merge(
    exposure,
    on=["dtname", "timeperiod"],
    how="inner"
)

df = df.merge(
    vulnerability,
    on=["dtname", "timeperiod"],
    how="inner"
)

df = df.merge(
    government_response,
    on=["dtname", "timeperiod"],
    how="inner"
)

# =============================================================================
# CLEAN KEYS
# =============================================================================

df["dtname"] = (
    df["dtname"]
    .astype(str)
    .str.strip()
)

df["timeperiod"] = (
    df["timeperiod"]
    .astype(str)
    .str.strip()
)

# =============================================================================
# TOPSIS WEIGHTS
# =============================================================================

weights = {
    "heat-hazard": 4,
    "exposure": 1,
    "vulnerability": 2,
    "government-response": 2,
}

total_weight = sum(weights.values())

# =============================================================================
# TOPSIS BY MONTH
# =============================================================================

results = []

for tp, g in df.groupby("timeperiod"):

    g = g.copy()

    norm = pd.DataFrame(index=g.index)

    # -------------------------------------------------------------------------
    # MIN-MAX NORMALIZATION
    # -------------------------------------------------------------------------

    for col in weights:

        min_v = g[col].min()
        max_v = g[col].max()

        if max_v == min_v:
            norm[col] = 0
        else:
            norm[col] = (
                (g[col] - min_v)
                /
                (max_v - min_v)
            )

    # -------------------------------------------------------------------------
    # APPLY WEIGHTS
    # -------------------------------------------------------------------------

    for col in weights:
        norm[col] = (
            norm[col]
            *
            (weights[col] / total_weight)
        )

    # -------------------------------------------------------------------------
    # IDEAL BEST / WORST
    # -------------------------------------------------------------------------

    ideal_best = norm.max()
    ideal_worst = norm.min()

    # -------------------------------------------------------------------------
    # DISTANCES
    # -------------------------------------------------------------------------

    dist_best = np.sqrt(
        ((norm - ideal_best) ** 2).sum(axis=1)
    )

    dist_worst = np.sqrt(
        ((norm - ideal_worst) ** 2).sum(axis=1)
    )

    # -------------------------------------------------------------------------
    # TOPSIS SCORE
    # -------------------------------------------------------------------------

    g["topsis-score"] = (
        dist_worst
        /
        (dist_best + dist_worst)
    )

    results.append(g)

# =============================================================================
# COMBINE RESULTS
# =============================================================================

df = pd.concat(
    results,
    ignore_index=True
)

# =============================================================================
# RISK CLASSIFICATION
# =============================================================================

def classify(score):

    if score <= 0.20:
        return 1
    elif score <= 0.40:
        return 2
    elif score <= 0.60:
        return 3
    elif score <= 0.80:
        return 4
    else:
        return 5


df["heat-risk-score"] = (
    df["topsis-score"]
    .apply(classify)
)

# =============================================================================
# SUMMARY
# =============================================================================

print("\nTOPSIS Summary")
print(df["topsis-score"].describe())

print("\nRisk Class Distribution")
print(
    df["heat-risk-score"]
    .value_counts()
    .sort_index()
)

print("\nPreview")
print(
    df[
        [
            "dtname",
            "timeperiod",
            "topsis-score",
            "heat-risk-score"
        ]
    ].head()
)

# =============================================================================
# APPEND TO MASTER VARIABLES
# =============================================================================

master = pd.read_csv(MASTER_CSV)

master.columns = (
    master.columns
    .str.replace("_", "-", regex=False)
)

master["district"] = (
    master["district"]
    .astype(str)
    .str.strip()
)

master["timeperiod"] = (
    master["timeperiod"]
    .astype(str)
    .str.strip()
)

# Remove old risk columns if already present

for col in [
    "heat-hazard",
    "exposure",
    "vulnerability",
    "government-response",
    "total-tender-awarded-value-fy-cumsum",
    "topsis-score",
    "heat-risk-score",
]:
    if col in master.columns:
        master = master.drop(columns=col)

# Merge risk outputs

final_df = master.merge(
    df[
        [
            "dtname",
            "timeperiod",
            "heat-hazard",
            "exposure",
            "vulnerability",
            "government-response",
            "total-tender-awarded-value-fy-cumsum",
            "topsis-score",
            "heat-risk-score",
        ]
    ],
    on=["dtname", "timeperiod"],
    how="left"
)


# =============================================================================
# SAVE RC LEVEL OUTPUT
# =============================================================================

OUTPUT = Path("data") / "final_risk_score.csv"

final_df.to_csv(
    OUTPUT,
    index=False
)

print(f"\nSaved RC-level file: {OUTPUT}")
print(f"Rows: {len(final_df)}")
print(f"Columns: {len(final_df.columns)}")

# =============================================================================
# DISTRICT LEVEL AGGREGATION
# =============================================================================

print("\nAvailable columns:")
print(final_df.columns.tolist())

district_df = (
    final_df.groupby(
        ["dtname", "timeperiod"],
        as_index=False
    )
    .agg(
        {
            "dtname": "first",
            "dtcode11": "first",     

            "health-centres-count": "sum",
            "total-tender-awarded-value": "sum",

            "sum-aged-population": "sum",
            "sum-young-population": "sum",
            "sum-population": "sum",

            "avg-electricity": "mean",

            "rc-piped-hhds-pct": "mean",
            "rc-nosanitation-hhds-pct": "mean",

            "workers-affected-pct": "mean",
            "pct-ncd": "mean",

            "land-surface-temperature": "mean",
            "land-surface-temperature-raster": "first",
            "heat-days-score": "mean",

            "heat-hazard": "mean",
            "exposure": "mean",
            "vulnerability": "mean",
            "government-response": "mean",
            "total-tender-awarded-value-fy-cumsum": "mean",

            "topsis-score": "mean",
            "heat-risk-score": "mean",
        }
    )
)

# renaming columns
district_df = district_df.rename(
    columns={"dtcode11": "object-id"}
)
district_df = district_df.rename(
    columns={"dtname": "district"}
)
district_df = district_df.rename(
    columns={"rc-piped-hhds-pct": "piped-hhds-pct"}
)
district_df = district_df.rename(
    columns={"rc-nosanitation-hhds-pct": "nosanitation-hhds-pct"}
)

# =============================================================================
# ROUND NUMERIC FIELDS
# =============================================================================

numeric_cols = district_df.select_dtypes(
    include=np.number
).columns

district_df[numeric_cols] = (
    district_df[numeric_cols]
    .round(3)
)

# =============================================================================
# SAVE DISTRICT OUTPUT
# =============================================================================

DISTRICT_OUTPUT = (
    Path("data")
    / "district_final_risk_score.csv"
)

district_df.to_csv(
    DISTRICT_OUTPUT,
    index=False
)

print(f"\nSaved district file: {DISTRICT_OUTPUT}")
print(f"Rows: {len(district_df)}")
print(f"Columns: {len(district_df.columns)}")

print("\nDistrict Preview")
print(
    district_df[
        [
            "district",
            "timeperiod",
            "topsis-score",
            "heat-risk-score"
        ]
    ].head()
)


TOPSIS Summary
count    2170.000000
mean        0.486402
std         0.174971
min         0.000000
25%         0.345141
50%         0.472136
75%         0.607719
max         1.000000
Name: topsis-score, dtype: float64

Risk Class Distribution
heat-risk-score
1     53
2    725
3    835
4    454
5    103
Name: count, dtype: int64

Preview
       dtname timeperiod  topsis-score  heat-risk-score
0      BAJALI    2021_04      0.277002                2
1       BAKSA    2021_04      0.277002                2
2     BARPETA    2021_04      0.338134                2
3   BISWANATH    2021_04      0.287903                2
4  BONGAIGAON    2021_04      0.306707                2



Saved RC-level file: data/final_risk_score.csv
Rows: 11160
Columns: 36

Available columns:
['object-id', 'dtname', 'rc-area', 'timeperiod', 'total-tender-awarded-value', 'heat-days-score', 'land-surface-temperature', 'year', 'sum-aged-population', 'sum-young-population', 'sum-population', 'health-centres-count', 'revenue-ci', 'workers-affected-pct', 'dtcode11', 'net-sown-area-in-hac', 'avg-electricity', 'avg-tele', 'rc-piped-hhds-pct', 'rc-nosanitation-hhds-pct', 'total-hhd', 'district', 'revenue-circle', 'women-sugar', 'men-sugar', 'women-bp', 'men-bp', 'pct-ncd', 'land-surface-temperature-raster', 'heat-hazard', 'exposure', 'vulnerability', 'government-response', 'total-tender-awarded-value-fy-cumsum', 'topsis-score', 'heat-risk-score']

Saved district file: data/district_final_risk_score.csv
Rows: 2170
Columns: 23

District Preview
  district timeperiod  topsis-score  heat-risk-score
0   BAJALI    2021_04         0.277              2.0
1   BAJALI    2021_05         0.000           